# Fund comparison backtests

This notebook compares candidate funds on three axes: ending value, volatility, and max drawdown. It is the backward-looking part of the decision process and should stay separate from the tax/account analysis.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

candidate_dirs = [Path.cwd(), Path.cwd() / 'copilot_analysis']
analysis_dir = next(path for path in candidate_dirs if (path / 'fund_analysis.py').exists())
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from fund_analysis import fund_summary_table, load_all_funds, score_funds

In [3]:
all_funds = load_all_funds()
all_funds.head()

,date,nav,daily_return,fund
0,2022-03-14,144.57,-0.0168,avanza_global
1,2022-03-15,147.20,0.0182,avanza_global
2,2022-03-16,148.52,0.0090,avanza_global
3,2022-03-17,149.77,0.0084,avanza_global
4,2022-03-18,151.52,0.0117,avanza_global


In [4]:
summary = fund_summary_table(all_funds)
summary

,fund,start_date,end_date,observations,cagr,volatility,max_drawdown,ending_multiple
0,evli,2019-03-07,2026-03-12,1672,0.301023,0.366296,-0.421352,6.333393
1,dnb_teknologi_s,2022-12-14,2026-03-12,784,0.241950,0.168476,-0.239585,2.018596
2,swedbank_robur_tech_a,2017-08-22,2026-03-12,2071,0.225285,0.225762,-0.345262,5.684532
3,ct_lux_global_tech,2017-08-29,2026-03-12,2069,0.209911,0.271785,-0.380908,5.084014
4,finserve,2019-02-25,2026-03-13,1758,0.188915,0.200800,-0.337225,3.383764
5,bgf_tech_a2,2010-02-16,2026-03-13,4016,0.154864,0.222312,-0.488931,10.110487
6,jpm_us_tech,2007-02-19,2026-03-13,4881,0.129180,0.256361,-0.599283,10.132390
7,avanza_zero,2019-02-22,2026-03-12,1757,0.127684,0.183501,-0.319196,2.333032
8,spiltan_aktiefond_investmentbolag,2017-07-25,2026-03-13,2154,0.126661,0.197656,-0.324147,2.799635
9,avanza_global,2022-03-14,2026-03-12,968,0.114681,0.155970,-0.242056,1.542920


In [5]:
scored = score_funds(summary)
scored[['fund', 'cagr', 'volatility', 'max_drawdown', 'composite_score']]

,fund,cagr,volatility,max_drawdown,composite_score
0,dnb_teknologi_s,0.241950,0.168476,-0.239585,0.923077
1,swedbank_robur_tech_a,0.225285,0.225762,-0.345262,0.634615
2,finserve,0.188915,0.200800,-0.337225,0.615385
3,avanza_global,0.114681,0.155970,-0.242056,0.615385
4,avanza_zero,0.127684,0.183501,-0.319196,0.615385
5,evli,0.301023,0.366296,-0.421352,0.557692
6,ct_lux_global_tech,0.209911,0.271785,-0.380908,0.519231
7,spiltan_aktiefond_investmentbolag,0.126661,0.197656,-0.324147,0.519231
8,bgf_tech_a2,0.154864,0.222312,-0.488931,0.423077
9,nordea_alfa,0.077227,0.174459,-0.365267,0.365385


In [9]:
candidate_funds = [
    'nordea_alfa',
    'avanza_zero',
    'avanza_global',
    'spiltan_aktiefond_investmentbolag',
    'swedbank_robur_tech_a',
    'dnb_teknologi_s',
]

common_window = (
    all_funds.loc[all_funds['fund'].isin(candidate_funds)]
    .pivot(index='date', columns='fund', values='nav')
    .dropna()
)
growth = common_window.div(common_window.iloc[0])
growth.tail()

fund,avanza_global,avanza_zero,dnb_teknologi_s,nordea_alfa,spiltan_aktiefond_investmentbolag,swedbank_robur_tech_a
date,,,,,,
2026-03-06,1.492734,1.543685,2.014417,1.324992,1.455388,1.930317
2026-03-09,1.494007,1.521993,2.015279,1.304051,1.425127,1.957111
2026-03-10,1.490056,1.565858,2.008120,1.345431,1.465014,1.949421
2026-03-11,1.499766,1.553764,2.020358,1.334031,1.447388,1.985185
2026-03-12,1.493739,1.554396,2.018596,1.325746,1.451058,1.964541


In [10]:
fig = px.line(
    growth.reset_index().melt(id_vars='date', var_name='fund', value_name='growth_of_1kr'),
    x='date',
    y='growth_of_1kr',
    color='fund',
    title='Normalized growth over common history window'
)
fig.show()

In [11]:
comparison = scored.loc[scored['fund'].isin(candidate_funds)].copy()
comparison = comparison.sort_values('composite_score', ascending=False)
comparison[['fund', 'cagr', 'volatility', 'max_drawdown', 'ending_multiple', 'composite_score']]

,fund,cagr,volatility,max_drawdown,ending_multiple,composite_score
0,dnb_teknologi_s,0.241950,0.168476,-0.239585,2.018596,0.923077
1,swedbank_robur_tech_a,0.225285,0.225762,-0.345262,5.684532,0.634615
3,avanza_global,0.114681,0.155970,-0.242056,1.542920,0.615385
4,avanza_zero,0.127684,0.183501,-0.319196,2.333032,0.615385
7,spiltan_aktiefond_investmentbolag,0.126661,0.197656,-0.324147,2.799635,0.519231
9,nordea_alfa,0.077227,0.174459,-0.365267,1.690219,0.365385


## Notes

Edit `candidate_funds` as the shortlist changes. The composite score is intentionally simple and should be treated as a ranking aid, not a final decision rule.